## Differentiable harmonic synthesizer

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import Audio

class HarmonicSynth(nn.Module):
    def __init__(
        self,
        sample_rate=16000,
        n_harmonics=5,
        duration=0.5
    ):
        super().__init__()
        self.sample_rate = sample_rate
        self.n_harmonics = n_harmonics
        # Time vector for the audio buffer
        self.register_buffer(
            'time',
            torch.linspace(0, duration, int(sample_rate * duration))
        )

    def forward(self, f0, amplitudes):
        # Create harmonic multiplier indices
        harmonics_idx = torch.arange(
            1, self.n_harmonics + 1,
            device=f0.device,
            dtype=torch.float32
        )
        
        # Compute instantaneous phase for each harmonic
        phases = (2.0 * torch.pi * harmonics_idx.view(1, 1, -1) 
        * f0.view(-1, 1, 1) * self.time.view(1, -1, 1))
        
        # Sinusoidal harmonics
        synth_harmonics = amplitudes * torch.sin(phases)

        # Sum across harmonics
        audio = synth_harmonics.sum(dim=-1)
        return audio


In [5]:
sr = 16000
duration = 1
n_harmonics = 5
time_steps = int(sr * duration)

# Initialize synthesizer
synth = HarmonicSynth(sample_rate=sr, n_harmonics=n_harmonics, duration=duration)

true_f0 = torch.tensor([500, 100])
true_amps = torch.tensor([[1.0, 0.5, 0.25, 0.125, 0.0625]]) # Exponentially decaying spectrum

target_audio = synth(
        true_f0,
        # Expand amplitudes across the time dimension
        true_amps.unsqueeze(1).expand(-1, time_steps, -1)
)
Audio(target_audio, rate=sr)

In [7]:
# Learnable parameter, Optimizer, and Loss function
learned_amps = nn.Parameter(torch.rand(1, n_harmonics) * 0.1)
optimizer = optim.Adam([learned_amps], lr=0.05)
criterion = nn.MSELoss()

In [8]:
# Start Gradient Descent Loop
for epoch in range(150):
    optimizer.zero_grad()

    # Expand learned amplitudes across the time dimension
    learned_amps_expanded = learned_amps.unsqueeze(1).expand(-1, time_steps, -1)

    # Forward pass: generate audio from current parameter estimates
    pred_audio = synth(
        true_f0,
        learned_amps_expanded
    )

    # Compute loss against the target waveform
    loss = criterion(pred_audio, target_audio)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 25 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.8f}")

Epoch 24 | Loss: 0.00555632
Epoch 49 | Loss: 0.00052270
Epoch 74 | Loss: 0.00009784
Epoch 99 | Loss: 0.00000625
Epoch 124 | Loss: 0.00000015
Epoch 149 | Loss: 0.00000002


In [9]:
Audio(pred_audio.detach().cpu().numpy(), rate=sr)